# Oxford-IIIT Pet 影像分類
- **Quiz1**：問題定義與資料切分（K-Fold）
- **Quiz2**：Plain CNN vs. Transfer Learning（**不使用資料擴增**，作為 Quiz3 的「擴增前」基準）
- **Quiz3**：資料擴增前後比較、Kernel 視覺化、XAI

輸出：圖片 `Outputs/figures/Q{1,2,3}_*.png`、模型 `Outputs/models/q{1,2,3}_*.pt`、表格 `Outputs/*.csv`

In [ ]:
import sys; sys.path.append("Codes")
import torch, pandas as pd
from PIL import Image
from functools import partial
import Quiz1 as q1, Quiz2 as q2, Quiz3 as q3
q1.set_seed()
print(q2.DEVICE, "| outputs ->", q1.OUT_DIR)

## Quiz1 — 問題定義、資料切分

In [ ]:
trainval_df, test_df, class_names = q1.build_dataframes()
q1.summarize(trainval_df, test_df, class_names)     # -> Q1_class_distribution.png
q1.show_samples(trainval_df)                         # -> Q1_samples.png

In [ ]:
N_FOLDS = 5
splits = q1.get_kfold_splits(trainval_df, N_FOLDS)
for k,(tr,va) in enumerate(splits):
    print(f"Fold {k}: train={len(tr)}, val={len(va)}")
FOLD = 0
loaders = partial(q1.make_loaders, trainval_df, test_df, splits, FOLD)

## Quiz2-1 Plain CNN（自行設計，無資料擴增）

In [ ]:
train_l, val_l, test_l = loaders(img_size=128, batch_size=64, augment="none")
plain = q2.PlainCNN(len(class_names))
print("Plain CNN params:", q2.count_params(plain))
plain, hist_plain, sec_plain = q2.train_model(plain, train_l, val_l, epochs=30, lr=2e-3)
q2.plot_history(hist_plain, "PlainCNN", tag="plain")          # -> Q2_plain_history.png
q1.save_model(plain, "q2_plain_cnn")                           # -> q2_plain_cnn.pt

In [ ]:
pred_plain, m_plain = q2.predict_testset(plain, test_l, test_df, class_names)
pred_plain.to_csv(q1.OUT_DIR/"q2_pred_plain.csv", index=False)
display(pred_plain.head(10))
display(q2.per_class_accuracy(pred_plain).head(10))

## Quiz2-2 經典 CNN Backbone（ResNet-18 Transfer Learning，無資料擴增）

In [ ]:
train_b, val_b, test_b = loaders(img_size=224, batch_size=64, augment="none")
resnet = q2.build_backbone("resnet18", len(class_names), freeze=False)
print("ResNet-18 params:", q2.count_params(resnet))
resnet, hist_res, sec_res = q2.train_model(resnet, train_b, val_b, epochs=10, lr=1e-3)
q2.plot_history(hist_res, "ResNet18", tag="resnet18")         # -> Q2_resnet18_history.png
q1.save_model(resnet, "q2_resnet18")                           # -> q2_resnet18.pt

In [ ]:
pred_res, m_res = q2.predict_testset(resnet, test_b, test_df, class_names)
pred_res.to_csv(q1.OUT_DIR/"q2_pred_resnet18.csv", index=False)
display(pred_res.head(10))

## Quiz2-3 ROC、Macro-AUC、Accuracy vs. 參數量

In [ ]:
auc_plain, _ = q2.plot_roc(m_plain["labels"], m_plain["probs"], class_names, "Plain CNN", tag="plain")
auc_res, _   = q2.plot_roc(m_res["labels"],   m_res["probs"],   class_names, "ResNet-18", tag="resnet18")
rows = [
 dict(name="PlainCNN", params=q2.count_params(plain)[0], top1=m_plain["top1"], top5=m_plain["top5"], macro_auc=auc_plain, train_sec=sec_plain),
 dict(name="ResNet18 (TL)", params=q2.count_params(resnet)[0], top1=m_res["top1"], top5=m_res["top5"], macro_auc=auc_res, train_sec=sec_res),
]
summary = q2.compare_models(rows)
summary.to_csv(q1.OUT_DIR/"q2_model_comparison.csv", index=False)

## 進階 A — Plain CNN 超參數實驗
> 每一組設定都是**重新初始化模型並重新訓練**（超參數必須在訓練前決定，無法套用在已訓練模型上），因此**不使用**上面 Quiz2 訓練好的模型。
> 設計：one-factor-at-a-time，base 與 Quiz2 主訓練相同（無擴增、lr=2e-3），每次只改一個因素；為節省時間只訓練 8 epochs、只用驗證集（fold 0）比較，不碰 test。表中各組之間可比較，與上面 30 epochs 的主模型不直接比較。

In [ ]:
def plain_loaders(augment, img_size):
    tr, va, _ = loaders(img_size=128, batch_size=64, augment=augment); return tr, va
exp_plain = q2.run_experiments("plain", q2.PLAIN_GRID, plain_loaders, epochs=8, img_size=128)

## 進階 B — 經典 Backbone 超參數實驗
> 同樣每組重新載入預訓練權重後重新訓練（5 epochs）。比較：凍結 vs 微調、學習率、優化器、資料擴增、Label Smoothing，以及不同 backbone（ResNet-18 / MobileNetV2 / ResNet-50）的精度與參數量取捨。

In [ ]:
def bb_loaders(augment, img_size):
    tr, va, _ = loaders(img_size=224, batch_size=64, augment=augment); return tr, va
exp_bb = q2.run_experiments("backbone", q2.BACKBONE_GRID, bb_loaders, epochs=5)

## Quiz3-1 Data Augmentation 前後比較
- **擴增前**：直接沿用 Quiz2 訓練好的 Plain CNN / ResNet-18（`pretrained=`，不重訓）。
- **擴增後**：相同超參數、相同 epoch，只改擴增策略，**必須重新訓練**（擴增發生在訓練階段）。
- 策略說明見下方 `AUG_RATIONALE`。

In [ ]:
print(q3.AUG_RATIONALE)
q3.show_aug_examples(trainval_df, ["none","geometric","photometric","full","trivialaug"], img_size=224)

In [ ]:
df_aug_plain, models_plain, hist_aug_plain = q3.run_aug_comparison(
    "PlainCNN", lambda: q2.PlainCNN(len(class_names)), trainval_df, test_df, splits, FOLD,
    policies=("none","geometric","full"), img_size=128, epochs=30, lr=2e-3,
    pretrained={"none": (plain, hist_plain, sec_plain)})

In [ ]:
df_aug_bb, models_bb, hist_aug_bb = q3.run_aug_comparison(
    "ResNet18", lambda: q2.build_backbone("resnet18", len(class_names)), trainval_df, test_df, splits, FOLD,
    policies=("none","geometric","full"), img_size=224, epochs=10, lr=1e-3,
    pretrained={"none": (resnet, hist_res, sec_res)})
pd.concat([df_aug_plain, df_aug_bb])

## Quiz3-2 Kernel 視覺化（使用 Quiz3 擴增後的 ResNet-18，第一層 7×7 卷積）
從 Quiz3 擴增後的模型（geometric / full）中，選驗證集 Top-1 最高者。

In [ ]:
aug_df = df_aug_bb[df_aug_bb.aug != "none"].set_index("aug")
best_aug = aug_df["val_top1"].idxmax()
resnet_q3 = models_bb[best_aug]
print("Quiz3 ResNet18 使用的擴增策略：", best_aug)

eval_tf224 = q1.get_transforms(224,"none")[1]
sample_img = eval_tf224(Image.open(test_df.iloc[0]["path"]).convert("RGB"))
q3.show_all_kernels(resnet_q3, f"ResNet18 (Quiz3, {best_aug})", tag="resnet18_q3")
stats, idxs = q3.visualize_two_kernels(resnet_q3, sample_img, title=f"ResNet18 (Quiz3, {best_aug})", tag="resnet18_q3")

## Quiz3-3 XAI（使用 Quiz3 擴增後的 ResNet-18）

In [ ]:
# 測試集預測（Quiz3 ResNet-18），用來挑選答對 / 答錯案例
pred_q3, m_q3 = q2.predict_testset(resnet_q3, test_b, test_df, class_names)
pred_q3.to_csv(q1.OUT_DIR/"q3_pred_resnet18.csv", index=False)
good, bad = q3.pick_cases(pred_q3)
q3.explain_samples(resnet_q3, test_df, class_names, good, 224, "ResNet18 (Quiz3) - correct", tag="resnet18_q3_correct")
q3.explain_samples(resnet_q3, test_df, class_names, bad,  224, "ResNet18 (Quiz3) - high-confidence errors", tag="resnet18_q3_errors")

In [ ]:
trimap_dir = q3.find_trimap_dir()
xai_q3,  summ_q3  = q3.xai_quantify(resnet_q3, test_df, class_names, trimap_dir, n=150, img_size=224)   # 擴增後
xai_q2,  summ_q2  = q3.xai_quantify(resnet,    test_df, class_names, trimap_dir, n=150, img_size=224)   # 擴增前(Quiz2)，對照用
for n_, d in [("resnet18_q3_aug", summ_q3), ("resnet18_q2_noaug", summ_q2)]:
    if len(d): d.to_csv(q1.OUT_DIR/f"q3_xai_{n_}.csv")

## 結論（跑完後填寫）
- Plain CNN vs. ResNet-18：Top-1 / Top-5 / Macro-AUC / 參數量 / 訓練時間
- 最佳超參數組合與原因
- 擴增前後差異（accuracy、train-val gap、Grad-CAM 前景集中度）
- 兩個 kernel 的功能解讀、易混淆品種的錯誤分析